# FastSpeech 2 Text-to-Speech (English & Telugu) — Final End-to-End Colab Pipeline

### Overview of the 7 Clean Cells
- **Cell 1 — Setup & Environment:** Mounts Google Drive, clones/updates the repository, installs all Python dependencies (`tgt`, `pyworld`, `g2p-en`, `datasets`), and extracts the Universal HiFi-GAN vocoder weights.
- **Cell 2 — English Data & Feature Pipeline (`ai4bharat/Svarah`):** Prepares the ~5h train / ~1h val split (`seed=42`), reuses verified MFA `.TextGrid` alignments from Drive if available (or runs MFA `english_us_arpa` alignment automatically), extracts pause-preserved features (`include_empty_intervals=True`), verifies feature counts, and backs up to Google Drive.
- **Cell 3 — Telugu Data & Feature Pipeline (`SPRINGLab/IndicTTS_Telugu`):** Prepares the ~5h train / ~1h val split (`seed=42`), reuses verified MFA `.TextGrid` alignments from Drive if available (or runs train-only MFA acoustic training + alignment automatically), extracts pause-preserved features, verifies feature counts, and backs up to Google Drive.
- **Cell 4 — Vocoder & Feature Quality Check:** Reconstructs ground-truth mel-spectrograms through Universal HiFi-GAN in isolated subprocesses and displays audio players for both languages.
- **Cell 5 — Train English FastSpeech 2 Model (60,000 Steps):** Trains the English model with real-time checkpoint & loss logging to Google Drive (auto-resumes if interrupted).
- **Cell 6 — Train Telugu FastSpeech 2 Model (60,000 Steps):** Trains the Telugu model with real-time checkpoint & loss logging to Google Drive (auto-resumes if interrupted).
- **Cell 7 — Generate All 10 Test Audios (`E1–E4`, `T1–T4`), 30 Variance Plots & Final Zip:** Synthesizes all 10 required test sentences, displays audio players for every sentence, verifies all plots/curves, and packages `Results_Deliverables_Final.zip` on Google Drive.

## Cell 1: Mount Google Drive, Clone/Update Repo, Install Dependencies & Extract HiFi-GAN Vocoder

In [ ]:
import os, sys, glob, shutil, subprocess
os.environ["MPLBACKEND"] = "Agg"

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
OLD_DRIVE_DIR = "/content/drive/MyDrive/FastSpeech2"
REPO_URL = "https://github.com/SandeshAwate6055/FastSpeech2-English-Telugu.git"
REPO_DIR = "/content/FastSpeech2-English-Telugu"

for sub in [
    "English/checkpoints",
    "English/logs",
    "Telugu/checkpoints",
    "Telugu/logs",
    "Results/English",
    "Results/Telugu",
    "diagnostics",
]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

# Remove any broken/empty (< 20 MB) tarballs created by earlier interrupted runs in FastSpeech2_CleanRun
for lang in ["English", "Telugu"]:
    tpath = os.path.join(BASE_DIR, lang, f"{lang}_preprocessed.tar.gz")
    if os.path.exists(tpath) and os.path.getsize(tpath) < 20 * 1024 * 1024:
        print(f"Removing incomplete archive {tpath} ({os.path.getsize(tpath)/(1024*1024):.2f} MB)")
        os.remove(tpath)

# Clone or update repository
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "--", "."], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Repository ready at commit: {commit}")

# Install Python packages into active kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tgt", "pyworld", "g2p-en", "datasets", "soundfile"], check=True)
subprocess.run([
    sys.executable, "-c",
    "import nltk; nltk.download('averaged_perceptron_tagger', quiet=True); "
    "nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('cmudict', quiet=True)"
], check=True)

# Extract Universal HiFi-GAN weights
for lang in ["English", "Telugu"]:
    hifi_ckpt = os.path.join(REPO_DIR, lang, "hifigan", "generator_universal.pth.tar")
    if not os.path.exists(hifi_ckpt):
        if not os.path.exists("/tmp/ming_fs2"):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ming024/FastSpeech2.git", "/tmp/ming_fs2"], check=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/ming_fs2/hifigan/generator_universal.pth.tar.zip", "-d", os.path.join(REPO_DIR, lang, "hifigan")], check=True)
        print(f"Extracted generator_universal.pth.tar for {lang}")

print("\nSUCCESS: Cell 1 Complete!")


## Cell 2: English Data Preparation, Alignment & Pause-Preserved Feature Extraction (`ai4bharat/Svarah`)

In [ ]:
import os, sys, glob, json, shutil, subprocess
from getpass import getpass
os.environ["MPLBACKEND"] = "Agg"
clean_env = {**os.environ, "MPLBACKEND": "Agg"}

BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
OLD_DRIVE_DIR = "/content/drive/MyDrive/FastSpeech2"
REPO_DIR = "/content/FastSpeech2-English-Telugu"
EN_DIR = os.path.join(REPO_DIR, "English")
EN_TAR = os.path.join(BASE_DIR, "English", "English_preprocessed.tar.gz")
OLD_EN_TAR = os.path.join(OLD_DRIVE_DIR, "English_preprocessed.tar.gz")

def ensure_mfa_installed():
    if not os.path.exists("/usr/local/bin/micromamba"):
        subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba && mv -f bin/micromamba /usr/local/bin/micromamba", shell=True, check=True)
    if not os.path.exists("/root/.local/share/mamba/envs/mfa"):
        subprocess.run([
            "/usr/local/bin/micromamba", "create", "-y", "-n", "mfa", "-c", "conda-forge",
            "montreal-forced-aligner=2.2.17", "openfst=1.8.2", "kaldi=5.5.1068", "joblib<1.4", "setuptools<81", "python=3.10"
        ], env=clean_env, check=True)

if os.path.exists(EN_TAR) and os.path.getsize(EN_TAR) > 50 * 1024 * 1024:
    print(f"Found validated English archive at {EN_TAR} ({os.path.getsize(EN_TAR)/(1024*1024):.1f} MB). Restoring ...")
    subprocess.run(["tar", "-xzf", EN_TAR, "-C", EN_DIR], check=True)
else:
    # 1. Prepare raw audio (~5h train, ~1h val) if not already prepared locally
    if not (os.path.exists(os.path.join(EN_DIR, "raw_data", "train_filelist.txt")) and os.path.exists(os.path.join(EN_DIR, "raw_data", "data_stats.json"))):
        if not os.environ.get("HF_TOKEN"):
            try:
                from google.colab import userdata
                os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
            except Exception:
                os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Read Token (hf_...) for ai4bharat/Svarah: ")
            clean_env["HF_TOKEN"] = os.environ["HF_TOKEN"]
        print("1. Running English prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42 ...")
        subprocess.run([sys.executable, "prepare_data.py", "--train_hours", "5.0", "--val_hours", "1.0", "--seed", "42"], cwd=EN_DIR, env=clean_env, check=True)
    else:
        print("1. Reusing existing English raw_data on local disk.")

    # 2. Obtain MFA TextGrids (fast-restore from existing Drive archive if available, else run MFA align)
    tg_dir = os.path.join(EN_DIR, "preprocessed_data", "TextGrid")
    existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))
    if len(existing_tgs) < 4000 and os.path.exists(OLD_EN_TAR):
        print(f"2. Extracting verified English TextGrids from {OLD_EN_TAR} ...")
        stage_en = "/content/stage_en_tg"
        shutil.rmtree(stage_en, ignore_errors=True)
        os.makedirs(stage_en, exist_ok=True)
        subprocess.run(["tar", "-xzf", OLD_EN_TAR, "-C", stage_en, "preprocessed_data/TextGrid"], check=False)
        staged_tg = os.path.join(stage_en, "preprocessed_data", "TextGrid")
        if os.path.exists(staged_tg):
            shutil.rmtree(tg_dir, ignore_errors=True)
            os.makedirs(os.path.dirname(tg_dir), exist_ok=True)
            shutil.copytree(staged_tg, tg_dir)
        shutil.rmtree(stage_en, ignore_errors=True)
        existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))

    if len(existing_tgs) < 4000:
        print("2. Running MFA forced alignment (english_us_arpa) ...")
        ensure_mfa_installed()
        subprocess.run(["/usr/local/bin/micromamba", "run", "-n", "mfa", "mfa", "model", "download", "acoustic", "english_us_arpa"], env=clean_env, check=True)
        subprocess.run([
            "/usr/local/bin/micromamba", "run", "-n", "mfa", "mfa", "align",
            os.path.join(EN_DIR, "raw_data"),
            os.path.join(EN_DIR, "lexicon", "english_svarah_mfa.dict"),
            "english_us_arpa",
            tg_dir,
            "--clean", "--single_speaker", "--num_jobs", "4"
        ], env=clean_env, check=True)
        existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))
    print(f"   Verified {len(existing_tgs)} English .TextGrid files ready.")

    # 3. Run pause-preserving preprocess.py
    for sub in ["mel", "pitch", "energy", "duration"]:
        shutil.rmtree(os.path.join(EN_DIR, "preprocessed_data", sub), ignore_errors=True)
    print("3. Running English preprocess.py --config config.yaml ...")
    subprocess.run([sys.executable, "preprocess.py", "--config", "config.yaml"], cwd=EN_DIR, env=clean_env, check=True)

    # 4. Verify output counts & archive to Google Drive
    train_lines = [l for l in open(os.path.join(EN_DIR, "preprocessed_data", "train.txt"), encoding="utf-8") if l.strip()]
    val_lines = [l for l in open(os.path.join(EN_DIR, "preprocessed_data", "val.txt"), encoding="utf-8") if l.strip()]
    assert len(train_lines) > 2500 and len(val_lines) > 500, f"Unexpectedly low clip counts: train={len(train_lines)}, val={len(val_lines)}"

    print(f"4. Backing up English features ({len(train_lines)} train, {len(val_lines)} val) to {EN_TAR} ...")
    subprocess.run([
        "tar", "-czf", EN_TAR, "-C", EN_DIR,
        "preprocessed_data", "lexicon", "raw_data/data_stats.json", "raw_data/train_filelist.txt", "raw_data/val_filelist.txt"
    ], check=True)

print("\nSUCCESS: Cell 2 (English Pipeline) Complete!")


## Cell 3: Telugu Data Preparation, Alignment & Pause-Preserved Feature Extraction (`SPRINGLab/IndicTTS_Telugu`)

In [ ]:
import os, sys, glob, json, shutil, subprocess
os.environ["MPLBACKEND"] = "Agg"
clean_env = {**os.environ, "MPLBACKEND": "Agg"}

BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
OLD_DRIVE_DIR = "/content/drive/MyDrive/FastSpeech2"
REPO_DIR = "/content/FastSpeech2-English-Telugu"
TE_DIR = os.path.join(REPO_DIR, "Telugu")
TE_TAR = os.path.join(BASE_DIR, "Telugu", "Telugu_preprocessed.tar.gz")
OLD_TE_TAR = os.path.join(OLD_DRIVE_DIR, "Telugu_preprocessed.tar.gz")

def ensure_mfa_installed():
    if not os.path.exists("/usr/local/bin/micromamba"):
        subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba && mv -f bin/micromamba /usr/local/bin/micromamba", shell=True, check=True)
    if not os.path.exists("/root/.local/share/mamba/envs/mfa"):
        subprocess.run([
            "/usr/local/bin/micromamba", "create", "-y", "-n", "mfa", "-c", "conda-forge",
            "montreal-forced-aligner=2.2.17", "openfst=1.8.2", "kaldi=5.5.1068", "joblib<1.4", "setuptools<81", "python=3.10"
        ], env=clean_env, check=True)

if os.path.exists(TE_TAR) and os.path.getsize(TE_TAR) > 50 * 1024 * 1024:
    print(f"Found validated Telugu archive at {TE_TAR} ({os.path.getsize(TE_TAR)/(1024*1024):.1f} MB). Restoring ...")
    subprocess.run(["tar", "-xzf", TE_TAR, "-C", TE_DIR], check=True)
else:
    # 1. Prepare raw audio (~5h train, ~1h val) if not already prepared locally
    if not (os.path.exists(os.path.join(TE_DIR, "raw_data", "train_filelist.txt")) and os.path.exists(os.path.join(TE_DIR, "raw_data", "data_stats.json"))):
        print("1. Running Telugu prepare_data.py --train_hours 5.0 --val_hours 1.0 --seed 42 ...")
        subprocess.run([sys.executable, "prepare_data.py", "--train_hours", "5.0", "--val_hours", "1.0", "--seed", "42"], cwd=TE_DIR, env=clean_env, check=True)
    else:
        print("1. Reusing existing Telugu raw_data on local disk.")

    # 2. Obtain MFA TextGrids (fast-restore from existing Drive archive if available, else run MFA train + align)
    tg_dir = os.path.join(TE_DIR, "preprocessed_data", "TextGrid")
    existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))
    if len(existing_tgs) < 2500 and os.path.exists(OLD_TE_TAR):
        print(f"2. Extracting verified Telugu TextGrids from {OLD_TE_TAR} ...")
        stage_te = "/content/stage_te_tg"
        shutil.rmtree(stage_te, ignore_errors=True)
        os.makedirs(stage_te, exist_ok=True)
        subprocess.run(["tar", "-xzf", OLD_TE_TAR, "-C", stage_te, "preprocessed_data/TextGrid"], check=False)
        staged_tg = os.path.join(stage_te, "preprocessed_data", "TextGrid")
        if os.path.exists(staged_tg):
            shutil.rmtree(tg_dir, ignore_errors=True)
            os.makedirs(os.path.dirname(tg_dir), exist_ok=True)
            shutil.copytree(staged_tg, tg_dir)
        shutil.rmtree(stage_te, ignore_errors=True)
        existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))

    if len(existing_tgs) < 2500:
        print("2. Running Telugu train-only MFA acoustic model training & alignment ...")
        ensure_mfa_installed()
        te_train_corpus = "/content/te_mfa_train_corpus"
        shutil.rmtree(te_train_corpus, ignore_errors=True)
        for spk in ["te_female", "te_male"]:
            os.makedirs(os.path.join(te_train_corpus, spk), exist_ok=True)
        for mf in ["train_filelist.txt", "reserve_train_filelist.txt"]:
            mf_path = os.path.join(TE_DIR, "raw_data", mf)
            if os.path.exists(mf_path):
                with open(mf_path, "r", encoding="utf-8") as f:
                    for line in f:
                        parts = line.strip().split("|")
                        if len(parts) >= 2:
                            uid, spk = parts[0], parts[1]
                            for ext in [".wav", ".lab"]:
                                src = os.path.join(TE_DIR, "raw_data", spk, uid + ext)
                                dst = os.path.join(te_train_corpus, spk, uid + ext)
                                if os.path.exists(src) and not os.path.exists(dst):
                                    os.symlink(src, dst)

        fast_mfa_cfg = (
            "beam: 10\nretry_beam: 40\nfeatures:\n  type: mfcc\n  use_energy: false\n  frame_shift: 10\n"
            "training:\n  - monophone:\n      num_iterations: 20\n      max_gaussians: 1000\n"
            "  - triphone:\n      num_iterations: 20\n      num_leaves: 2000\n      max_gaussians: 10000\n"
            "  - lda:\n      num_iterations: 20\n      num_leaves: 2500\n      max_gaussians: 15000\n"
        )
        with open("/content/fast_mfa_train.yaml", "w", encoding="utf-8") as f:
            f.write(fast_mfa_cfg)

        ac_zip = os.path.join(TE_DIR, "te_acoustic.zip")
        subprocess.run([
            "/usr/local/bin/micromamba", "run", "-n", "mfa", "mfa", "train",
            te_train_corpus,
            os.path.join(TE_DIR, "lexicon", "telugu-lexicon.txt"),
            ac_zip,
            "--config_path", "/content/fast_mfa_train.yaml",
            "--single_speaker", "--clean", "--num_jobs", "4"
        ], env=clean_env, check=True)

        subprocess.run([
            "/usr/local/bin/micromamba", "run", "-n", "mfa", "mfa", "align",
            os.path.join(TE_DIR, "raw_data"),
            os.path.join(TE_DIR, "lexicon", "telugu-lexicon.txt"),
            ac_zip,
            tg_dir,
            "--single_speaker", "--clean", "--num_jobs", "4"
        ], env=clean_env, check=True)
        existing_tgs = glob.glob(os.path.join(tg_dir, "*", "*.TextGrid"))
    print(f"   Verified {len(existing_tgs)} Telugu .TextGrid files ready.")

    # 3. Run pause-preserving preprocess.py
    for sub in ["mel", "pitch", "energy", "duration"]:
        shutil.rmtree(os.path.join(TE_DIR, "preprocessed_data", sub), ignore_errors=True)
    print("3. Running Telugu preprocess.py --config config.yaml ...")
    subprocess.run([sys.executable, "preprocess.py", "--config", "config.yaml"], cwd=TE_DIR, env=clean_env, check=True)

    # 4. Verify output counts & archive to Google Drive
    train_lines = [l for l in open(os.path.join(TE_DIR, "preprocessed_data", "train.txt"), encoding="utf-8") if l.strip()]
    val_lines = [l for l in open(os.path.join(TE_DIR, "preprocessed_data", "val.txt"), encoding="utf-8") if l.strip()]
    assert len(train_lines) > 1500 and len(val_lines) > 300, f"Unexpectedly low clip counts: train={len(train_lines)}, val={len(val_lines)}"

    print(f"4. Backing up Telugu features ({len(train_lines)} train, {len(val_lines)} val) to {TE_TAR} ...")
    subprocess.run([
        "tar", "-czf", TE_TAR, "-C", TE_DIR,
        "preprocessed_data", "lexicon", "raw_data/data_stats.json", "raw_data/train_filelist.txt", "raw_data/val_filelist.txt"
    ], check=True)

print("\nSUCCESS: Cell 3 (Telugu Pipeline) Complete!")


## Cell 4: Data Integrity Summary & HiFi-GAN Vocoder Audio Verification

In [ ]:
import os, sys, json, subprocess
from IPython.display import Audio, display
os.environ["MPLBACKEND"] = "Agg"

REPO_DIR = "/content/FastSpeech2-English-Telugu"
BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
DIAG_DIR = os.path.join(BASE_DIR, "diagnostics")
os.makedirs(DIAG_DIR, exist_ok=True)

voc_script = r'''
import os, sys, json, yaml, torch
import numpy as np
from scipy.io import wavfile

lang = sys.argv[1]
lang_dir = sys.argv[2]
out_dir = sys.argv[3]
sys.path.insert(0, lang_dir)

from utils.model import get_vocoder, vocoder_infer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = yaml.safe_load(open(os.path.join(lang_dir, "config.yaml"), encoding="utf-8"))
vocoder = get_vocoder(cfg, device)

pdir = os.path.join(lang_dir, "preprocessed_data")
train_lines = [l.strip().split("|") for l in open(os.path.join(pdir, "train.txt"), encoding="utf-8") if l.strip()]
val_lines = [l.strip().split("|") for l in open(os.path.join(pdir, "val.txt"), encoding="utf-8") if l.strip()]
sp_count = sum(1 for parts in train_lines if "sp" in parts[2].strip("{}").split())

uid, spk, ph_str, raw_txt = train_lines[0][0], train_lines[0][1], train_lines[0][2], train_lines[0][3]
mel = np.load(os.path.join(pdir, "mel", f"{spk}-mel-{uid}.npy"))
mel_t = torch.from_numpy(mel).float().unsqueeze(0).transpose(1, 2).to(device)
wav = vocoder_infer(mel_t, vocoder, cfg, cfg)[0]
wav_path = os.path.join(out_dir, f"{lang}_{uid}_vocoder_check.wav")
wavfile.write(wav_path, 22050, wav)

info = {
    "train_clips": len(train_lines),
    "val_clips": len(val_lines),
    "train_clips_with_sp_pauses": sp_count,
    "sample_uid": uid,
    "sample_spk": spk,
    "sample_text": raw_txt,
    "sample_wav": wav_path,
}
with open(os.path.join(out_dir, f"{lang}_vocoder_check.json"), "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
'''
with open("/tmp/voc_check.py", "w", encoding="utf-8") as f:
    f.write(voc_script)

for lang in ["English", "Telugu"]:
    lang_dir = os.path.join(REPO_DIR, lang)
    tar_path = os.path.join(BASE_DIR, lang, f"{lang}_preprocessed.tar.gz")
    if not os.path.exists(os.path.join(lang_dir, "preprocessed_data", "train.txt")) and os.path.exists(tar_path):
        subprocess.run(["tar", "-xzf", tar_path, "-C", lang_dir], check=True)

    subprocess.run([sys.executable, "/tmp/voc_check.py", lang, lang_dir, DIAG_DIR], check=True)
    info = json.load(open(os.path.join(DIAG_DIR, f"{lang}_vocoder_check.json"), encoding="utf-8"))
    print(f"\n=== {lang} Summary ===")
    print(f"  Train clips: {info['train_clips']} (with 'sp' interior pauses: {info['train_clips_with_sp_pauses']}) | Val clips: {info['val_clips']}")
    print(f"  Vocoder Sample: {info['sample_uid']} ({info['sample_spk']}) -> {info['sample_text'][:90]}")
    display(Audio(info["sample_wav"]))


## Cell 5: Train English FastSpeech 2 Model (60,000 Steps, Auto-Resumes from Drive)

In [ ]:
import os, sys, glob, subprocess
os.environ["MPLBACKEND"] = "Agg"
clean_env = {**os.environ, "MPLBACKEND": "Agg"}

REPO_DIR = "/content/FastSpeech2-English-Telugu"
BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
EN_DIR = os.path.join(REPO_DIR, "English")
EN_CKPT = os.path.join(BASE_DIR, "English", "checkpoints")
EN_LOG = os.path.join(BASE_DIR, "English", "logs")
EN_RES = os.path.join(BASE_DIR, "Results", "English")

for d in [EN_CKPT, EN_LOG, EN_RES]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(os.path.join(EN_DIR, "preprocessed_data", "train.txt")):
    en_tar = os.path.join(BASE_DIR, "English", "English_preprocessed.tar.gz")
    assert os.path.exists(en_tar), "Run Cell 2 first!"
    subprocess.run(["tar", "-xzf", en_tar, "-C", EN_DIR], check=True)

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(EN_CKPT, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"=== Training English FastSpeech 2 Model from step {restore_step} -> 60000 ===")
if restore_step < 60000:
    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", str(restore_step),
        "--total_step", "60000",
        "--val_step", "1000",
        "--save_step", "5000",
        "--ckpt_path", EN_CKPT,
        "--log_path", EN_LOG,
        "--result_path", EN_RES,
    ], cwd=EN_DIR, env=clean_env, check=True)
print("\nSUCCESS: English Model Training Complete!")


## Cell 6: Train Telugu FastSpeech 2 Model (60,000 Steps, Auto-Resumes from Drive)

In [ ]:
import os, sys, glob, subprocess
os.environ["MPLBACKEND"] = "Agg"
clean_env = {**os.environ, "MPLBACKEND": "Agg"}

REPO_DIR = "/content/FastSpeech2-English-Telugu"
BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"
TE_DIR = os.path.join(REPO_DIR, "Telugu")
TE_CKPT = os.path.join(BASE_DIR, "Telugu", "checkpoints")
TE_LOG = os.path.join(BASE_DIR, "Telugu", "logs")
TE_RES = os.path.join(BASE_DIR, "Results", "Telugu")

for d in [TE_CKPT, TE_LOG, TE_RES]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(os.path.join(TE_DIR, "preprocessed_data", "train.txt")):
    te_tar = os.path.join(BASE_DIR, "Telugu", "Telugu_preprocessed.tar.gz")
    assert os.path.exists(te_tar), "Run Cell 3 first!"
    subprocess.run(["tar", "-xzf", te_tar, "-C", TE_DIR], check=True)

ckpts = [
    int(os.path.basename(f).replace(".pth.tar", ""))
    for f in glob.glob(os.path.join(TE_CKPT, "*.pth.tar"))
    if os.path.basename(f).replace(".pth.tar", "").isdigit()
]
restore_step = max(ckpts) if ckpts else 0
print(f"=== Training Telugu FastSpeech 2 Model from step {restore_step} -> 60000 ===")
if restore_step < 60000:
    subprocess.run([
        sys.executable, "train.py",
        "--config", "config.yaml",
        "--restore_step", str(restore_step),
        "--total_step", "60000",
        "--val_step", "1000",
        "--save_step", "5000",
        "--ckpt_path", TE_CKPT,
        "--log_path", TE_LOG,
        "--result_path", TE_RES,
    ], cwd=TE_DIR, env=clean_env, check=True)
print("\nSUCCESS: Telugu Model Training Complete!")


## Cell 7: Synthesize All 10 Mandatory Test Sentences (`E1–E4`, `T1–T4`), Generate 30 Variance Plots & Package Final Zip

In [ ]:
import os, sys, json, glob, subprocess
from IPython.display import Audio, display
os.environ["MPLBACKEND"] = "Agg"
clean_env = {**os.environ, "MPLBACKEND": "Agg"}

REPO_DIR = "/content/FastSpeech2-English-Telugu"
BASE_DIR = "/content/drive/MyDrive/FastSpeech2_CleanRun"

for lang, spk_id in [("English", -1), ("Telugu", 0)]:
    lang_dir = os.path.join(REPO_DIR, lang)
    ckpt_dir = os.path.join(BASE_DIR, lang, "checkpoints")
    res_dir = os.path.join(BASE_DIR, "Results", lang)
    os.makedirs(res_dir, exist_ok=True)

    best_ckpt = os.path.join(ckpt_dir, "best_val.pth.tar")
    cmd = [
        sys.executable, "inference.py",
        "--config", "config.yaml",
        "--speaker_id", str(spk_id),
        "--ckpt_path", ckpt_dir,
        "--result_path", res_dir,
    ]
    if os.path.exists(best_ckpt):
        cmd.extend(["--ckpt_file", best_ckpt])

    print(f"\n==================== Synthesizing {lang} Test Sentences ====================")
    subprocess.run(cmd, cwd=lang_dir, env=clean_env, check=True)

    summary_path = os.path.join(res_dir, "variance_predictions_summary.json")
    summary = json.load(open(summary_path, encoding="utf-8"))
    for sid, item in summary.items():
        dur_s = item["total_frames"] * 256.0 / 22050.0
        print(f"\n[{lang} - {sid}] (Step {item['checkpoint_step']}, Speaker {item['speaker_id']}, {dur_s:.2f}s): {item['raw_text']}")
        wav_path = os.path.join(res_dir, "test_generated_audios", f"{sid}.wav")
        display(Audio(wav_path))

zip_path = os.path.join(BASE_DIR, "Results_Deliverables_Final.zip")
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(["zip", "-r", zip_path, "Results", "English/logs", "Telugu/logs"], cwd=BASE_DIR, check=True)
print(f"\nSUCCESS: Packaged all deliverables to {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.2f} MB)!")
